# 📊 Retail Sales Data Cleaning & Feature Engineering Notebook

This Jupyter notebook demonstrates end-to-end data cleaning, missing value imputation, duplicate removal, data type casting, filtering, and feature engineering using **Pandas** and **NumPy**.

In [ ]:
%pip install -q pandas numpy

import pandas as pd
import numpy as np

# Set display options for better visibility
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

## Step 1: Read the Raw CSV File
Read `sales_data.csv` specifying `header=2` because the first 2 rows contain metadata headers.

In [ ]:
df_raw = pd.read_csv('sales_data.csv', header=2)
print('Initial Dataset Shape:', df_raw.shape)
df_raw.head()

## Step 2: Clean and Rename Columns
Standardize column names: strip leading/trailing whitespace, convert to lowercase, and replace spaces with underscores.

In [ ]:
df = df_raw.copy()
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
print('Cleaned Column Names:', df.columns.tolist())
df.head()

## Step 3: Drop Completely Empty Rows & Cast Numeric Data Types
Remove rows where all values are NaN, and ensure numeric columns are properly converted to numerical data types.

In [ ]:
rows_all_null = df.isnull().all(axis=1)
print(f'Completely empty rows found: {rows_all_null.sum()}')

df = df.dropna(how='all').reset_index(drop=True)

# Convert numeric columns
numeric_cols = ['order_id', 'quantity', 'unit_price', 'sales', 'profit', 'discount']
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

df.head()

## Step 4: Drop Unnecessary Columns
**Justification:** Drop column `'city'` as it contains missing values and is non-essential for financial order & revenue analysis.

In [ ]:
df = df.drop(columns=['city'])
df.head()

## Step 5: Check Missing Values Count
Count missing values across all remaining columns.

In [ ]:
missing_counts = df.isnull().sum()
print('Missing values per column:')
print(missing_counts)

## Step 6: Impute & Handle Missing Numeric Values
1. Drop rows where critical value `'sales'` is missing.
2. Fill missing `'profit'` with the **mean** profit.
3. Fill missing `'discount'` with the **median** discount.

In [ ]:
# Drop missing sales
sales_missing_count = df['sales'].isnull().sum()
df = df.dropna(subset=['sales']).reset_index(drop=True)
print(f'Dropped {sales_missing_count} row(s) missing sales.')

# Impute profit and discount
profit_mean = df['profit'].mean()
discount_median = df['discount'].median()

print(f'Mean profit calculated: {profit_mean:.2f}')
print(f'Median discount calculated: {discount_median:.2f}')

df['profit'] = df['profit'].fillna(profit_mean)
df['discount'] = df['discount'].fillna(discount_median)

print('\nMissing values remaining in numeric columns:')
print(df[['sales', 'profit', 'discount']].isnull().sum())

## Step 7: Handle Missing Categorical Data
Fill missing `'customer_name'` values with `'Unknown'`.

In [ ]:
missing_cust_before = df['customer_name'].isnull().sum()
df['customer_name'] = df['customer_name'].fillna('Unknown')
print(f'Replaced {missing_cust_before} missing customer_name value(s) with Unknown.')

## Step 8: Detect & Remove Duplicates
Identify duplicate orders based on `order_id` and keep only the first occurrence.

In [ ]:
duplicates_mask = df.duplicated(subset=['order_id'], keep=False)
print('Duplicate rows identified:')
display(df[duplicates_mask][['order_id', 'customer_name', 'category', 'product', 'sales']])

num_duplicates = df.duplicated(subset=['order_id'], keep='first').sum()
df = df.drop_duplicates(subset=['order_id'], keep='first').reset_index(drop=True)
print(f'\nRemoved {num_duplicates} duplicate row(s) based on order_id.')

## Step 9: Data Filtering & Feature Engineering
1. **Filter 1:** Orders with `unit_price > 20,000`.
2. **Filter 2:** Completed orders with `unit_price > 10,000`.
3. **Feature 1 (`total_amount`):** `quantity * unit_price - discount`
4. **Feature 2 (`customer_type`):** `'Bulk Buyer'` if `quantity >= 3` else `'Regular Buyer'`

In [ ]:
print('--- Filter 1: Orders with unit_price > 20,000 ---')
high_unit_price = df[df['unit_price'] > 20000]
display(high_unit_price[['order_id', 'customer_name', 'product', 'unit_price']])

print('\n--- Filter 2: Completed orders with unit_price > 10,000 ---')
filter2 = df[(df['unit_price'] > 10000) & (df['status'] == 'Completed')]
display(filter2[['customer_name', 'category', 'status', 'unit_price']])

# Create new feature columns
df['total_amount'] = df['quantity'] * df['unit_price'] - df['discount']
df['customer_type'] = np.where(df['quantity'] >= 3, 'Bulk Buyer', 'Regular Buyer')

print('\nSample rows with new feature columns:')
display(df[['order_id', 'customer_name', 'quantity', 'unit_price', 'discount', 'total_amount', 'customer_type']].head())

## Step 10: Final Clean Dataset Verification & Export
Compare dataset shape before and after cleaning, inspect final output, and export to CSV.

In [ ]:
print(f'Dataset Shape BEFORE cleaning: {df_raw.shape}')
print(f'Dataset Shape AFTER cleaning:  {df.shape}')

print('\nFirst 10 rows of final cleaned dataset:')
display(df.head(10))

# Export clean dataset to CSV
df.to_csv('cleaned_sales_data.csv', index=False)
print('\nCleaned dataset saved to cleaned_sales_data.csv.')